In [47]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import rdkit
import gdown

In [48]:
# # 1. Get the URL from the 'Share' button in Drive
# url = 'https://drive.google.com/file/d/1uVl3dKKGAvyoW_53R-L4abTns3AQfbmy/view?usp=sharing'

# # 2. Download the file
# # 'fuzzy=True' allows gdown to extract the File ID automatically from the link
# output = 'dataset.csv'
# gdown.download(url, output, quiet=False, fuzzy=True)

# # 3. Load with Pandas
# import pandas as pd
# df = pd.read_csv(output)

In [49]:
df = pd.read_csv("tox21.csv")
print("Data load: size - ", df.shape)

Data load: size -  (7831, 14)


In [50]:
df.head(5)

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [51]:
df.columns

Index(['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
       'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
       'mol_id', 'smiles'],
      dtype='str')

In [52]:
print(f"total number of null values= {df.isnull().sum().sum()}")

total number of null values= 16026


In [53]:
df.isnull().sum()

NR-AR             566
NR-AR-LBD        1073
NR-AhR           1282
NR-Aromatase     2010
NR-ER            1638
NR-ER-LBD         876
NR-PPAR-gamma    1381
SR-ARE           1999
SR-ATAD5          759
SR-HSE           1364
SR-MMP           2021
SR-p53           1057
mol_id              0
smiles              0
dtype: int64

In [54]:
targets = [
'NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase',
'NR-ER','NR-ER-LBD','NR-PPAR-gamma',
'SR-ARE','SR-ATAD5','SR-HSE','SR-MMP','SR-p53'
]

df[targets] = df[targets].fillna(0)

In [55]:
from rdkit import Chem
from rdkit.Chem import Descriptors

def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    return [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol)
    ]
features = df['smiles'].apply(smiles_to_features)

# remove invalid rows
df = df[features.notnull()]
features = features[features.notnull()]
print(">> shape: ", df.shape)

X = list(features)
features[:5]

[05:04:23] WARNING: not removing hydrogen atom without neighbors
[05:04:26] Explicit valence for atom # 8 Al, 6, is greater than permitted
[05:04:27] Explicit valence for atom # 3 Al, 6, is greater than permitted
[05:04:27] Explicit valence for atom # 4 Al, 6, is greater than permitted
[05:04:29] Explicit valence for atom # 4 Al, 6, is greater than permitted
[05:04:31] Explicit valence for atom # 9 Al, 6, is greater than permitted
[05:04:31] Explicit valence for atom # 5 Al, 6, is greater than permitted
[05:04:33] Explicit valence for atom # 16 Al, 6, is greater than permitted
[05:04:35] Explicit valence for atom # 20 Al, 6, is greater than permitted


>> shape:  (7823, 14)


0         [258.32399999999996, 1.3424, 1, 5, 82.28, 3]
1    [204.22899999999998, 1.2993999999999999, 1, 2,...
2         [288.475, 5.090300000000005, 1, 1, 20.23, 1]
3    [276.42400000000004, 3.7524400000000018, 1, 2,...
4    [206.027, -0.9922000000000002, 5, 3, 135.29000...
Name: smiles, dtype: object

In [56]:
feature_cols = [
    'MolWt', 'LogP', 'HDonors', 'HAcceptors', 'TPSA', 'RotBonds'
]

features_df = pd.DataFrame(features.tolist(), columns=feature_cols)
features_df.index = df.index

# X = features_df

# targets = [
#     'NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase',
#     'NR-ER','NR-ER-LBD','NR-PPAR-gamma',
#     'SR-ARE','SR-ATAD5','SR-HSE','SR-MMP','SR-p53'
# ]

# y = df[targets]

df = pd.concat([df, features_df], axis=1)

In [57]:
df.columns

Index(['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD',
       'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53',
       'mol_id', 'smiles', 'MolWt', 'LogP', 'HDonors', 'HAcceptors', 'TPSA',
       'RotBonds'],
      dtype='str')

In [45]:
X.head()

,MolWt,LogP,HDonors,HAcceptors,TPSA,RotBonds
0,258.324,1.34240,1,5,82.28,3
1,204.229,1.29940,1,2,49.41,2
2,288.475,5.09030,1,1,20.23,1
3,276.424,3.75244,1,2,32.34,7
4,206.027,-0.99220,5,3,135.29,2


In [58]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
# create generator once
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    fp = fpgen.GetFingerprint(mol)
    arr = np.array(fp)
    return arr

fps = df['smiles'].apply(smiles_to_fp)

df = df[fps.notnull()]
fps = fps[fps.notnull()]

fp_df = pd.DataFrame(fps.tolist())
fp_df.index = df.index

df = pd.concat([df, fp_df], axis=1)

[05:05:03] WARNING: not removing hydrogen atom without neighbors


In [61]:
df.columns

Index([        'NR-AR',     'NR-AR-LBD',        'NR-AhR',  'NR-Aromatase',
               'NR-ER',     'NR-ER-LBD', 'NR-PPAR-gamma',        'SR-ARE',
            'SR-ATAD5',        'SR-HSE',
       ...
                  1014,            1015,            1016,            1017,
                  1018,            1019,            1020,            1021,
                  1022,            1023],
      dtype='object', length=1044)

In [62]:
targets = [
    'NR-AR','NR-AR-LBD','NR-AhR','NR-Aromatase',
    'NR-ER','NR-ER-LBD','NR-PPAR-gamma',
    'SR-ARE','SR-ATAD5','SR-HSE','SR-MMP','SR-p53'
]

y = df[targets]

X = df.drop(columns=targets + ['mol_id', 'smiles'])
X.columns = X.columns.astype(str)

print("X shape:", X.shape)
print("y shape:", y.shape)
print(X.columns[:10])   # just to inspect

X shape: (7823, 1030)
y shape: (7823, 12)
Index(['MolWt', 'LogP', 'HDonors', 'HAcceptors', 'TPSA', 'RotBonds', '0', '1',
       '2', '3'],
      dtype='str')


In [63]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [64]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=200,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1
    )
)

model.fit(X_train, y_train)

,estimator estimator: estimator objectAn estimator object implementing :term:`fit` and :term:`predict`.A :term:`predict_proba` method will be exposed only if `estimator` implementsit.,RandomForestC...ndom_state=42)
,"n_jobs n_jobs: int or None, optional (default=None)The number of jobs to run in parallel.:meth:`fit`, :meth:`predict` and :meth:`partial_fit` (if supportedby the passed estimator) will be parallelized for each target.When individual estimators are fast to train or predict,using ``n_jobs > 1`` can result in slower performance dueto the parallelism overhead.``None`` means `1` unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all available processes / threads.See :term:`Glossary ` for more details... versionchanged:: 0.20 `n_jobs` default changed from `1` to `None`.",None
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the f

In [65]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

d:\MyProjects\drug_toxicity_pred\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
d:\MyProjects\drug_toxicity_pred\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
d:\MyProjects\drug_toxicity_pred\venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
d:\MyProjects\drug_toxicity_pred\venv\Lib\site-packages\sklea

In [66]:
from sklearn.metrics import roc_auc_score

print("ROC-AUC Scores:\n")

for i, col in enumerate(targets):
    try:
        prob = y_prob[i][:, 1]
        score = roc_auc_score(y_test[col], prob)
        print(f"{col}: {score:.3f}")
    except:
        print(f"{col}: skipped")

ROC-AUC Scores:

NR-AR: 0.798
NR-AR-LBD: 0.853
NR-AhR: 0.873
NR-Aromatase: 0.756
NR-ER: 0.758
NR-ER-LBD: 0.800
NR-PPAR-gamma: 0.887
SR-ARE: 0.790
SR-ATAD5: 0.867
SR-HSE: 0.809
SR-MMP: 0.857
SR-p53: 0.861


In [ ]:
print(X_test)

        MolWt     LogP  HDonors  HAcceptors    TPSA  RotBonds  0  1  2  3  \
1142   75.067 -0.97030        2           2   63.32         1  0  0  0  0   
4262  404.410  2.87950        3           6   91.68         9  0  1  0  0   
2167  318.373  1.62290        0           4   66.92         5  0  0  0  0   
1678  397.491  1.95120        2           5   89.82         8  0  1  0  1   
3537  226.276  1.38940        2           2   78.76         4  0  1  0  0   
...       ...      ...      ...         ...     ...       ... .. .. .. ..   
3919  602.641 -0.45420        9          12  242.98        10  0  1  0  0   
4323  118.132  1.17940        0           3   35.53         2  0  0  0  0   
7354  338.521 -0.70450        2           3   49.69         3  0  0  0  0   
5763  125.127  0.62928        0           3   50.09         3  0  0  0  0   
5013  622.539  0.65020        5          11  228.51        12  0  0  0  0   

      ...  1014  1015  1016  1017  1018  1019  1020  1021  1022  1023  
114

### prediction pipeline

In [67]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, rdFingerprintGenerator
# fingerprint generator (same as before)
fpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)

def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError("Invalid SMILES string")

    # Descriptors
    desc = [
        Descriptors.MolWt(mol),
        Descriptors.MolLogP(mol),
        Descriptors.NumHDonors(mol),
        Descriptors.NumHAcceptors(mol),
        Descriptors.TPSA(mol),
        Descriptors.NumRotatableBonds(mol)
    ]

    # Fingerprint
    fp = fpgen.GetFingerprint(mol)
    fp_array = np.array(fp)

    # Combine
    features = np.concatenate([desc, fp_array])

    return features

def prepare_input(smiles, X_columns):
    features = smiles_to_features(smiles)

    # convert to DataFrame
    input_df = pd.DataFrame([features], columns=X_columns)

    return input_df

def predict_smiles(smiles, model, X_columns, targets):

    input_df = prepare_input(smiles, X_columns)

    preds = model.predict(input_df)
    probs = model.predict_proba(input_df)

    result = {}

    for i, col in enumerate(targets):
        result[col] = {
            "prediction": int(preds[0][i]),
            "probability": float(probs[i][0][1])
        }

    return result

X_columns = X.columns


In [ ]:
smiles = "C1=CC=C(C=C1)O"

output = predict_smiles(smiles, model, X_columns, targets)

print(output)

{'NR-AR': {'prediction': 0, 'probability': 0.0}, 'NR-AR-LBD': {'prediction': 0, 'probability': 0.005}, 'NR-AhR': {'prediction': 0, 'probability': 0.04}, 'NR-Aromatase': {'prediction': 0, 'probability': 0.005}, 'NR-ER': {'prediction': 0, 'probability': 0.045}, 'NR-ER-LBD': {'prediction': 0, 'probability': 0.075}, 'NR-PPAR-gamma': {'prediction': 0, 'probability': 0.005}, 'SR-ARE': {'prediction': 0, 'probability': 0.06}, 'SR-ATAD5': {'prediction': 0, 'probability': 0.03}, 'SR-HSE': {'prediction': 0, 'probability': 0.01}, 'SR-MMP': {'prediction': 0, 'probability': 0.075}, 'SR-p53': {'prediction': 0, 'probability': 0.01}}
